# Data Preprocessing & Feature Engineering

This notebook prepares the **IBM HR Attrition dataset.** dataset for machine
learning. The analysis steps live in `01_eda.ipynb`.

**Objectives**
- Remove unnecessary / leakage columns
- Fix incorrect data types and handle missing values
- Engineer new features
- Encode categorical features and scale numerical ones
- Split into train/test and persist a reusable preprocessing pipeline


In [37]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

DATA_RAW = "../data/raw"
DATA_PROCESSED = "../data/processed"
MODELS_DIR = "../models"

RAW_FILE = os.path.join(DATA_RAW, "HR-Employee-Attrition.csv")
CLEANED_FILE = os.path.join(DATA_PROCESSED, "cleaned_employee-attrition.csv")
X_TRAIN_RAW = os.path.join(DATA_PROCESSED, "X_train_raw.csv")
X_TEST_RAW = os.path.join(DATA_PROCESSED, "X_test_raw.csv")
X_TRAIN_CSV = os.path.join(DATA_PROCESSED, "X_train.csv")
X_TEST_CSV = os.path.join(DATA_PROCESSED, "X_test.csv")
Y_TRAIN_CSV = os.path.join(DATA_PROCESSED, "y_train.csv")
Y_TEST_CSV = os.path.join(DATA_PROCESSED, "y_test.csv")
PREPROCESSOR_PKL = os.path.join(MODELS_DIR, "preprocessor.pkl")

In [38]:
df = pd.read_csv(RAW_FILE)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [40]:
print(df.shape)

(1470, 35)


In [41]:
df.columns.tolist()

['Age',
 'Attrition',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EmployeeCount',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'Over18',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StandardHours',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [42]:
df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

In [43]:
df.dtypes

Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears   

In [44]:
df_clean = df.copy()

In [45]:
print(df.shape)

print(df_clean.shape)

(1470, 35)
(1470, 35)


In [46]:
columns_to_drop =[
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"

]


In [47]:
columns_to_drop

['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']

In [48]:
df_clean.drop(columns=columns_to_drop, inplace=True)

In [49]:
df_clean.shape

(1470, 31)

In [50]:
missing = df_clean.isnull().sum()
missing

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithCurrManager        0
dtype: int64

In [51]:
print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

print(f"Rows           : {df_clean.shape[0]}")
print(f"Columns        : {df_clean.shape[1]}")
print(f"Missing Values : {df_clean.isnull().sum().sum()}")
print(f"Duplicate Rows : {df_clean.duplicated().sum()}")

print("\nData Types\n")
print(df_clean.dtypes.value_counts())

DATASET VALIDATION
Rows           : 1470
Columns        : 31
Missing Values : 0
Duplicate Rows : 0

Data Types

int64     23
object     8
Name: count, dtype: int64


In [52]:
target = "Attrition"

numerical_features = [
'Age',
 'DailyRate',
 'DistanceFromHome',
 'HourlyRate',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'PercentSalaryHike',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

categorical_features = [
'BusinessTravel',
 'Department',
 'EducationField',
 'Gender',
 'JobRole',
 'MaritalStatus',
 'OverTime']

ordinal_features =[
 'Education',
 'EnvironmentSatisfaction',
 'JobInvolvement',
 'JobLevel',
 'JobSatisfaction',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StockOptionLevel',
 'WorkLifeBalance']

In [53]:
print("Numerical Features   :", len(numerical_features))
print("Categorical Features :", len(categorical_features))
print("ordinal Features :", len(ordinal_features))

Numerical Features   : 14
Categorical Features : 7
ordinal Features : 9


In [54]:
X = df_clean.drop(columns=[target])
y = df_clean[target]

In [55]:
y = df_clean[target].map({"Yes": 1, "No": 0})

In [56]:
y.value_counts()

Attrition
0    1233
1     237
Name: count, dtype: int64

In [57]:
df_clean.to_csv(CLEANED_FILE, index=False)

# Train / Test Split

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [59]:
print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Training Shape : (1176, 30)
Testing Shape  : (294, 30)
Attrition
0    0.838435
1    0.161565
Name: proportion, dtype: float64
Attrition
0    0.840136
1    0.159864
Name: proportion, dtype: float64


In [60]:
X_train.to_csv(X_TRAIN_RAW, index=False)
X_test.to_csv(X_TEST_RAW, index=False)

# Build Preprocessing Pipeline

## Numeric, Categorical & Ordinal Pipelines

In [61]:
# No missing values remain, but the imputer keeps the pipeline
# defensive if new data arrives with missing entries.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [62]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [63]:
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [64]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("ord", ordinal_pipeline, ordinal_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [65]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

In [66]:
print(X_train_processed.shape)

print(X_test_processed.shape)

(1176, 51)
(294, 51)


# Save Pipeline

In [67]:
joblib.dump(preprocessor, PREPROCESSOR_PKL)

['../models\\preprocessor.pkl']

# Save Processed Dataset

In [68]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

In [69]:
X_train_processed.to_csv(X_TRAIN_CSV, index=False)
X_test_processed.to_csv(X_TEST_CSV, index=False)

y_train.to_csv(Y_TRAIN_CSV, index=False)
y_test.to_csv(Y_TEST_CSV, index=False)

In [70]:
files = [
    X_TRAIN_CSV,
    X_TEST_CSV,
    Y_TRAIN_CSV,
    Y_TEST_CSV,
    PREPROCESSOR_PKL
]

for file in files:
    print(file, ":", os.path.exists(file))

../data/processed\X_train.csv : True
../data/processed\X_test.csv : True
../data/processed\y_train.csv : True
../data/processed\y_test.csv : True
../models\preprocessor.pkl : True


# Final Validation

In [71]:
summary = {
    "Original Rows": df.shape[0],
    "Original Columns": df.shape[1],
    "Final Rows": df_clean.shape[0],
    "Final Columns": df_clean.shape[1],
    "Missing Values": int(df_clean.isnull().sum().sum()),
    "Duplicate Rows": int(df_clean.duplicated().sum()),
    "Numerical Features": len(numerical_features),
    "Categorical Features": len(categorical_features),
    "Ordinal Features": len(ordinal_features)
}

pd.DataFrame(summary.items(), columns=["Metric", "Value"])

,Metric,Value
0,Original Rows,1470
1,Original Columns,35
2,Final Rows,1470
3,Final Columns,31
4,Missing Values,0
5,Duplicate Rows,0
6,Numerical Features,14
7,Categorical Features,7
8,Ordinal Features,9


In [72]:
print("=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print(f"Training Samples  : {X_train.shape[0]}")
print(f"Testing Samples   : {X_test.shape[0]}")

print(f"Original Features  : {X.shape[1]}")
print(f"Processed Features : {X_train_processed.shape[1]}")

print("\nMissing Values")
print(X_train_processed.isnull().sum().sum())

print("\nPipeline Saved")
print(os.path.exists(PREPROCESSOR_PKL))

PREPROCESSING COMPLETED
Training Samples  : 1176
Testing Samples   : 294
Original Features  : 30
Processed Features : 51

Missing Values
0

Pipeline Saved
True
